# 04 - Language Models
## Module B: Language Understanding and Generation

Covers extractive summarization, semantic search / query expansion via embeddings, and automatic insight generation.> **Setup note:** This notebook uses the `src/` package from the project root.
> If running in Google Colab, first mount/clone the repo so `src/` is on the path,
> and place the Kaggle `BBC News Train.csv` in `data/raw/` for the real dataset
> (otherwise the bundled offline sample in `data/sample/` is used automatically).


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

from src.data_processing.data_loader import load_news_data, dataset_source
print("Dataset source:", dataset_source())


Dataset source: Bundled sample dataset (demo only): /home/claude/ITAI2373-NewsBot-Final/data/sample/sample_news.csv


In [2]:
df = pd.read_csv('../data/processed/articles_processed.csv') if os.path.exists('../data/processed/articles_processed.csv') else load_news_data()
df.shape

(200, 10)

## Intelligent summarization (extractive TextRank)

In [3]:
from src.language_models.summarizer import Summarizer

summarizer = Summarizer()
sample = df.iloc[0]
print("ORIGINAL:\n", sample['content'], "\n")
print("SUMMARY:\n", summarizer.summarize(sample['content'], n_sentences=2))

ORIGINAL:
 Atlas Energy announced layoffs affecting hundreds of employees as it restructures its airline division. Pinnacle Airlines announced layoffs affecting hundreds of employees as it restructures its banking division. Meridian Bank announced layoffs affecting hundreds of employees as it restructures its manufacturing division. 

SUMMARY:
 Atlas Energy announced layoffs affecting hundreds of employees as it restructures its airline division. Pinnacle Airlines announced layoffs affecting hundreds of employees as it restructures its banking division.


## Semantic search over the corpus

In [4]:
from src.language_models.embeddings import EmbeddingIndex

index = EmbeddingIndex()
index.build(df['content'].tolist())

hits = index.most_similar("technology company announces new product", top_k=5)
for h in hits:
    print(f"[{h['score']:.3f}] {h['document'][:90]}...")

[0.247] Shares of TechCorp rose sharply after the firm announced a new product line. The retail se...
[0.241] Atlas Energy unveiled its latest device, featuring unprecedented processing power that the...
[0.187] Regulators are examining Atlas Energy over concerns related to immigration policy in the m...
[0.175] Regulators are examining BrightWave Media over concerns related to tax reform in the airli...
[0.166] The central bank held interest rates steady, saying inflation pressures tied to rising ene...


## Query expansion (improves recall for short queries)

In [5]:
expanded = index.expand_query("election campaign")
print("Expanded query terms:", expanded)

Expanded query terms: ['campaign', 'clinched', 'fought', 'celebrated', 'fans', 'title']


## Content enhancement / automatic insight generation

In [6]:
from src.analysis.sentiment_analyzer import SentimentAnalyzer
from src.analysis.ner_extractor import NERExtractor
from src.language_models.generator import InsightGenerator

sa, ner, gen = SentimentAnalyzer(), NERExtractor(), InsightGenerator()

article = df.iloc[5]
sentiment = sa.analyze(article['content'])
entities = ner.extract_entities(article['content'])

enhanced = gen.enhance_article({
    'category': article['category'],
    'sentiment_label': sentiment['label'],
    'entities': entities,
    'confidence': 0.87,
})
print(enhanced['enhancement'])

This article was classified as **tech** (confidence: 87%). Its overall tone is **positive**. Organizations involved: Orion Motors, AI, Meridian Bank.


## Aggregate business insights

In [7]:
from src.utils.export import build_summary_report

df_sent = sa.analyze_dataframe(df, text_col='content')
summary = build_summary_report(df_sent)
for insight in gen.generate_business_insights(summary):
    print("-", insight)

- 'tech' is the most represented category, accounting for 40 of 200 articles.
- 'entertainment' coverage skews most positive (avg sentiment 0.82), while 'politics' skews most negative (avg sentiment 0.25).


## Key Takeaways
- Summarization uses TextRank (graph-based extractive method) - fast, deterministic, and fully offline; an optional transformer-based abstractive backend is stubbed in `summarizer.py` for environments with internet/HuggingFace access.
- Semantic search uses TF-IDF cosine similarity by default; `embeddings.py` also supports a sentence-transformers backend as a drop-in upgrade.
- `InsightGenerator` turns raw analysis into human-readable, business-facing narrative text.